# Basic usage
## Inference
The tensors are bf16. The CPU reference is computed in fp32 from the same bf16 values, and the
results are compared at bf16's precision (rtol = atol = 2e-2).

In [ ]:
import os
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config.yml")
os.environ["TORCHSIM_LOG_PATH"] = os.path.join(os.getcwd(), "togsim_results")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "outputs")
import torch

### Normal Matmul Code

In [ ]:
torch.manual_seed(0)
input = torch.randn(128, 128, dtype=torch.bfloat16)
weight = torch.randn(128, 128, dtype=torch.bfloat16)

opt_fn = torch.compile(torch.matmul)
cpu_output = opt_fn(input.float(), weight.float())

### PyTorchSim Matmul Code

In [ ]:
device = torch.device("npu:0")

npu_output = opt_fn(input.to(device), weight.to(device))
print(npu_output.dtype, npu_output.device)

In [ ]:
def test_result(name, npu_output, cpu_output, rtol=2e-2, atol=2e-2, preview=5):
    """bf16 on the NPU against the fp32 reference computed from the same bf16 values."""
    npu_output_cpu = npu_output.cpu().float()
    cpu_output = cpu_output.float()

    if torch.allclose(npu_output_cpu, cpu_output, rtol=rtol, atol=atol):
        message = f"|{name} Functionality Test Passed|"
        print("-" * len(message))
        print(message)
        print("-" * len(message))

        print(f"npu output (first {preview}): {npu_output_cpu.flatten()[:preview]} ...")
        print(f"cpu output (first {preview}): {cpu_output.flatten()[:preview]} ...")
    else:
        message = f"|{name} Functionality Test Failed|"
        print("-" * len(message))
        print(message)
        print("-" * len(message))
        print("max abs diff: ", float((npu_output_cpu - cpu_output).abs().max()))

In [ ]:
test_result("MatMul", npu_output, cpu_output)